# Small-Scale Experiments

### Imports


In [1]:
import cupy
import matplotlib.pyplot as plt
import numpy as np
import s3fs
import zarr

from deep_neurographs.utils import img_util, swc_util, util
from scipy.nd import gaussian_laplace, gaussian_filter, maximum_filter, center_of_mass
from sk.feature import peak_local_max
from sk.measure import label
from sk.segmentation import watershed
from time import time

%matplotlib inline

IMG_PREFIX = {
    "685221": "exaSPIM_685221_2024-04-12_11-46-38_fusion_2024-07-22_21-00-15",
    "703070": "exaSPIM_703070_2024-07-09_12-52-33_flatfield-correction_2024-07-18_12-43-27_fusion_2024-08-02_17-11-59",
    "706301": "exaSPIM_706301_2024-04-23_11-24-24_fusion_2024-05-21_00-00-03",
    "708369": "exaSPIM_708369_2024-04-08_15-20-36_fusion_2024-05-20_23-30-43",
    "709393": "exaSPIM_709393_2024-04-17_09-37-51_fusion_2024-07-27_00-20-20",
    "715345": "exaSPIM_715345_2024-06-07_10-03-37_fusion_2024-07-02_10-30-40"
}

ModuleNotFoundError: No module named 'scipy.nd'

In [ ]:
# Subroutines
def rescale(arr, clip_bool=True):
    if clip_bool:
        arr = np.clip(arr, 0, np.percentile(arr, 99))
    arr -= np.min(arr)
    arr = (2**16 - 1) * (arr / np.max(arr))
    return (arr).astype(np.uint16)


def get_mip(arr, axis=0, clip_bool=False):
    mip = np.max(arr, axis=axis)
    mip = rescale(mip, clip_bool=clip_bool)
    return mip


def get_img_patch(img, voxel, shape, from_center=True):
    start, end = img_util.get_start_end(voxel, shape, from_center=from_center)
    return img[0, 0, start[2]:end[2], start[1]:end[1], start[0]:end[0]]


def plot_mips(volume, prefix="", clip_bool=False):
    fig, axs = plt.subplots(1, 3, figsize=(10, 4))
    axs_names = ["XY", "XZ", "YZ"]
    for i in range(3):
        axs[i].imshow(get_mip(volume, axis=i, clip_bool=clip_bool))
        axs[i].set_title(prefix + axs_names[i], fontsize=16)
        axs[i].set_xticks([])
        axs[i].set_yticks([])
    plt.tight_layout()
    plt.show()


def mark_voxel(img_patch, voxel, value=1000):
    i, j, k = voxel
    img_patch[i-2:i+1, j-2:j+1, k-2:k+1] = value
    return img_patch


def find_argmax_in_neighborhood(img, xyz, k):
    x, y, z = xyz
    half_k = k // 2
    
    # Define the neighborhood bounds
    x_min, x_max = max(x - half_k, 0), min(x + half_k + 1, img.shape[0])
    y_min, y_max = max(y - half_k, 0), min(y + half_k + 1, img.shape[1])
    z_min, z_max = max(z - half_k, 0), min(z + half_k + 1, img.shape[2])
    
    # Extract the kxkxk neighborhood around the point
    nbhd = img[x_min:x_max, y_min:y_max, z_min:z_max]
    
    # Find the argmax within the neighborhood
    local_argmax = np.unravel_index(np.argmax(nbhd), nbhd.shape)
    
    # Convert local argmax to global coordinates in the original img
    global_argmax = (local_argmax[0] + x_min, local_argmax[1] + y_min, local_argmax[2] + z_min)
    
    return global_argmax


### Opem img

In [ ]:
# Parameters
s3_bucket = "aind-open-data"
dataset = "706301"
downsample_factor = 4

# Initializations
fs = s3fs.S3FileSystem()
s3_url = f"s3://{s3_bucket}/{IMG_PREFIX[dataset]}/fused.zarr/{downsample_factor}/"

# Open img
store = s3fs.S3Map(root=s3_url, s3=fs)
img = zarr.open(store, mode='r')
print("img.shape:", img.shape)

### Read img

In [ ]:
traceable_somas = [
    [18403.324, 11245.411, 2675.9985],
    [23425.596, 8926.942, 2185.401],
    [18400.334,11231.872,2683.3584],
    [21624.133,7740.173,12768.359],
    [22883.316,4323.8574,16284.648],
    [43937.27,5479.154,12344.152],
]

bright_nonsomas = [
    [20026.29,6470.747,14384.427],
    [18330.354,3822.579,14477.964],
    [22800.148,2251.1611,16850.379],
    [37846.895,6913.5713,20431.992],
    [43469.51,4820.5303,16769.32],
    [44225.082,4402.9487,15589.198],
    [22086.203,16967.01,14625.586],
    [20239.145,21295.611,11517.228],
]

In [ ]:
# Region of interest
idx = 4
shape = [128, 128, 128]
from_center = True

# Read img
voxel = img_util.to_voxels([29021.332,13788.292,10055.257], downsample_factor=downsample_factor)
img_chunk = get_img_patch(img, voxel, shape, from_center=from_center)
#img_chunk = gaussian_filter(img_chunk, [1, 1, 1])

# Compute LoG
LoG_img = gaussian_laplace(img_chunk, 3)
LoG_thresholded_img = np.logical_and(
    LoG_img == maximum_filter(LoG_img, 3),
    LoG_img > 100,
)
thresholded_img = np.logical_and(LoG_thresholded_img, img_chunk > 100)


# Find blob centroids
labels, n_labels = label(thresholded_img, return_num=True)
centers = center_of_mass(img_chunk, labels=labels, index=np.arange(1, n_labels+1))
candidates = list()
value = np.max(img_chunk) + 200
for center in centers:
    center = tuple([int(c) for c in center])
    center = find_argmax_in_neighborhood(img_chunk, center, 6)
    img_chunk = mark_voxel(img_chunk, center, 200)
    candidates.append(center)


# Plot mips
print("# Objects Detected:", len(candidates))
plot_mips(img_chunk)
plot_mips(LoG_img)


In [ ]:
stop

## Large-Scale Experiment

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.spatial import KDTree
from scipy.spatial.distance import euclidean as dist
from tqdm import tqdm

import os
import random


def sliding_window_coords_3d(img, window_size, overlap):
    # Calculate the stride based on the overlap and window size
    stride = tuple(w - o for w, o in zip(window_size, overlap))
    
    # Get dimensions of the  and window
    _, _, z_dim, y_dim, x_dim = img.shape
    z_win, y_win, x_win = window_size
    z_stride, y_stride, x_stride = stride

    # Loop over the  with the sliding window
    coords = []
    for x in range(0, x_dim - x_win + 1, x_stride):
        for y in range(0, y_dim - y_win + 1, y_stride):
            for z in range(0, z_dim - z_win + 1, z_stride):
                coords.append((x, y, z))
    return coords


def find_candidates(img, offset, window_size):
    # Read patch
    img_patch = get_img_patch(img, offset, window_size, from_center=False)
    if np.max(img_patch) < 100:
        return list()

    # Find blobs
    LoG_img = gaussian_laplace(img_patch, 3)
    LoG_thresholded_img = np.logical_and(
        LoG_img == maximum_filter(LoG_img, 3),
        LoG_img > 100,
    )
    thresholded_img = np.logical_and(LoG_thresholded_img, img_patch > 100)

    # Get centroids
    labels, n_labels = label(thresholded_img, return_num=True)
    index = np.arange(1, n_labels + 1)
    centers = center_of_mass(img_patch, labels=labels, index=index)
    return filter_centers(img_patch, centers, offset)


def filter_centers(img_patch, centers, offset):
    filtered_centers = list()
    for center in centers:
        voxel = tuple([int(c) for c in center])
        voxel = find_argmax_in_neighborhood(img_patch, center, 6)
        if img_patch[voxel] > 100:
            xyz = local_voxel_to_physical(voxel[::-1], offset)
            filtered_centers.append(xyz)
    return filtered_centers


def local_voxel_to_physical(local_voxel, offset):
    global_voxel = np.array([v + o for v, o in zip(local_voxel, offset)])
    return img_util.to_world(global_voxel * 2 ** downsample_factor)


def avg_nearby_centers(centers, d=16):
    filtered_centers = list()
    kdtree = KDTree(centers)
    visited = set()
    for xyz in centers:
        if tuple(xyz) not in visited:
            idxs = kdtree.query_ball_point(xyz, d)
            center = average_centers(kdtree.data[idxs], visited)
            filtered_centers.append(center)
    return filtered_centers


def average_centers(xyz_list, visited):
    centers = list()
    for xyz in xyz_list:
        visited.add(tuple(xyz))
    return np.mean(xyz_list, axis=0)


### Soma Candidate Detection

In [ ]:
# Parameters
window_size = (128, 128, 128)
overlap = (32, 32, 32)

# Initializations
coords = sliding_window_coords_3d(img, window_size, overlap)
output_dir = "/root/capsule/results/soma_candidates"
util.mkdir(output_dir, delete=True)
print("# img patches:", len(coords))

# Detection
with ThreadPoolExecutor() as executor:
    # Assign threads
    threads = list()
    for coord in coords:  #random.sample(coords, 1000):
        threads.append(
            executor.submit(find_candidates, img, coord, window_size)
        )

    # Process results
    centers = list()
    pbar = tqdm(total=len(threads))
    for thread in as_completed(threads):
        centers.extend(thread.result())
        pbar.update(1)

print("# Candidates Detected before filtering:", len(centers))
candidates = avg_nearby_centers(centers, d=16)
print("# Candidates Detected after filtering:", len(candidates))

# Save somas
with ThreadPoolExecutor() as executor:
    threads = list()
    for i, xyz in enumerate(candidates):
        path = os.path.join(output_dir, f"{i + 1}.swc")
        threads.append(
            executor.submit(
                swc_util.save_point, path, xyz, 30, color="1.0 0.0 0.0"
            )
        )

    pbar = tqdm(total=len(threads))
    for _ in as_completed(threads):
        pbar.update(1)


In [ ]:
len(candidates)